In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/binary-classification-with-a-bank-dataset-clone/sample_submission.csv
/kaggle/input/binary-classification-with-a-bank-dataset-clone/train.csv
/kaggle/input/binary-classification-with-a-bank-dataset-clone/test.csv


In [ ]:
# ==========================
# 1️⃣ Library Imports
# ==========================
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================
# 2️⃣ Load Data
# ==========================
train = pd.read_csv("/kaggle/input/binary-classification-with-a-bank-dataset-clone/train.csv").drop('id', axis=1)
test = pd.read_csv("/kaggle/input/binary-classification-with-a-bank-dataset-clone/test.csv").drop('id', axis=1)

# ==========================
# 3️⃣ Feature Engineering
# ==========================
def apply_feature_engineering(df):
    df_fe = df.copy()
    
    # Label encoding for categorical features
    for col in df_fe.select_dtypes(include='object').columns:
        if col != 'y':
            le = LabelEncoder()
            df_fe[col] = le.fit_transform(df_fe[col])
    
    # Interaction features
    df_fe['poutcome_duration'] = df_fe['poutcome'] * df_fe['duration']
    df_fe['housing_duration'] = df_fe['housing'] * df_fe['duration']
    df_fe['poutcome_housing'] = df_fe['poutcome'] * df_fe['housing']
    
    # Duration-based features
    df_fe['duration_per_campaign'] = df_fe['duration'] / (df_fe['campaign'] + 1)
    df_fe['duration_log'] = np.log1p(df_fe['duration'])
    df_fe['is_long_call'] = (df_fe['duration'] > df_fe['duration'].median()).astype(int)
    
    # Job-based statistics
    df_fe['job_duration_mean'] = df_fe.groupby('job')['duration'].transform('mean')
    df_fe['job_age_mean'] = df_fe.groupby('job')['age'].transform('mean')
    df_fe['job_balance_mean'] = df_fe.groupby('job')['balance'].transform('mean')
    
    # Education-based statistics
    df_fe['education_duration_mean'] = df_fe.groupby('education')['duration'].transform('mean')
    df_fe['education_balance_mean'] = df_fe.groupby('education')['balance'].transform('mean')
    
    # Marital-based statistics
    df_fe['marital_age_mean'] = df_fe.groupby('marital')['age'].transform('mean')
    
    # Financial situation features
    df_fe['total_loans'] = df_fe['housing'] + df_fe['loan']
    df_fe['financial_stress'] = (df_fe['default'] + df_fe['housing'] + df_fe['loan']).clip(0, 3)
    
    # Contact pattern features
    df_fe['contact_success_rate'] = df_fe['poutcome'] / (df_fe['previous'] + 1)
    
    # Age-based features
    df_fe['age_balance_ratio'] = df_fe['balance'] / (df_fe['age'] + 1)
    
    return df_fe

train_fe = apply_feature_engineering(train)
test_fe = apply_feature_engineering(test)

# ==========================
# 4️⃣ Mutual Information Feature Importance
# ==========================
X = train_fe.drop(columns=['y'])
y = train_fe['y']

mi = mutual_info_classif(X, y, discrete_features='auto')
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)
print("Mutual Information Scores (after feature engineering):")
print(mi_series)

# ==========================
# 5️⃣ Hyperparameter Tuning (optional, can skip if using tuned models)
# ==========================
kf = StratifiedKFold(n_splits=5, shuffle=True)

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'max_bin': trial.suggest_int('max_bin', 256, 15000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.5, 2.0),
        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'gpu_hist',
        'verbosity': -1
    }
    model = xgb.XGBClassifier(**params)
    score = cross_val_score(model, X, y, cv=kf, scoring='roc_auc', n_jobs=1).mean()
    return score

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 800, 1500),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'max_bin': trial.suggest_int('max_bin', 5000, 7000),
        'num_leaves': trial.suggest_int('num_leaves', 30, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 2.0),
        'device': 'gpu',
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1
    }
    model = lgb.LGBMClassifier(**params)
    score = cross_val_score(model, X, y, cv=kf, scoring='roc_auc', n_jobs=1).mean()
    return score

# ==========================
# 6️⃣ Target Encoding + CV-Based Blending
# ==========================
def run_te_cvens_blending(train_data, test_data, submission_path, save_path, n_splits=10, xgb_model=None, lgb_model=None):
    X = train_data.drop(columns=["y"]).copy()
    y = train_data["y"]
    cat_cols = X.select_dtypes("object").columns.tolist()
    
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True)
    CV_result = []
    test_preds = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), 1):
        X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        test_fold = test_data.copy()
        
        # Target Encoding
        for col in cat_cols:
            encoding_dict = y_train.groupby(X_train[col]).mean().to_dict()
            global_mean = y_train.mean()
            for category in X_train[col].unique():
                n = (X_train[col] == category).sum()
                smooth_mean = (encoding_dict.get(category, global_mean) * n + global_mean * 5) / (n + 5)
                encoding_dict[category] = smooth_mean
            X_train[col] = X_train[col].map(encoding_dict).fillna(global_mean)
            X_valid[col] = X_valid[col].map(encoding_dict).fillna(global_mean)
            test_fold[col] = test_fold[col].map(encoding_dict).fillna(global_mean)
        
        # Models
        xgb_clf = xgb_model if xgb_model else xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss")
        lgb_clf = lgb_model if lgb_model else lgb.LGBMClassifier(objective="binary", verbose=-1)
        
        # Fit models
        xgb_clf.fit(X_train, y_train)
        lgb_clf.fit(X_train, y_train)
        
        # Predict probabilities
        y_pred_proba = (xgb_clf.predict_proba(X_valid)[:,1] + lgb_clf.predict_proba(X_valid)[:,1]) / 2.0
        CV_result.append({"fold": fold, "roc_auc": roc_auc_score(y_valid, y_pred_proba)})
        
        # Test predictions
        test_pred_fold = (xgb_clf.predict_proba(test_fold)[:,1] + lgb_clf.predict_proba(test_fold)[:,1]) / 2.0
        test_preds.append(test_pred_fold)
    
    CV_result = pd.DataFrame(CV_result)
    print(CV_result)
    print(f"Mean CV Score: {CV_result['roc_auc'].mean():.5f}")
    
    y_test_pred_proba = np.mean(test_preds, axis=0)
    submission = pd.read_csv(submission_path)
    submission["y"] = y_test_pred_proba
    submission.to_csv(save_path, index=False)

# ==========================
# 7️⃣ Tuned Models & Run
# ==========================
xgb_tuned = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    n_estimators=1389,
    max_bin=10847,
    max_depth=8,
    learning_rate=0.05834,
    subsample=0.87453,
    colsample_bytree=0.61278,
    reg_alpha=0.00389,
    reg_lambda=0.00375,
    min_child_weight=4,
    gamma=0.19473,
    scale_pos_weight=1.32847,
    grow_policy="lossguide",
    tree_method="hist"
)

lgb_tuned = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    n_estimators=1200,
    learning_rate=0.08247,
    num_leaves=110,
    max_depth=12,
    min_child_samples=10,
    subsample=0.74892,
    colsample_bytree=0.30928,
    reg_alpha=1.47382,
    reg_lambda=1.89473,
    max_bin=6000,
    verbosity=-1
)

run_te_cvens_blending(
    train_data=train_fe,
    test_data=test_fe,
    submission_path="/kaggle/input/binary-classification-with-a-bank-dataset-clone/sample_submission.csv",
    save_path="submission.csv",
    xgb_model=xgb_tuned,
    lgb_model=lgb_tuned
)
